# 06b — 模拟 XRD、热力学凸包与高通量虚拟筛选

上一节 `06a` 我们学会了用 pymatgen 把晶体结构表示出来、做对称性分析、从 COD 抓真实结构。但表示本身不是目的。本节要回答材料信息学最实际的三个问题：

1. **我合成出来的到底是什么相？**——把计算结构变成**模拟 XRD 谱图**，与实验衍射图对上号。这是连接“纸上的结构”和“炉子里的粉末”的桥梁；
2. **这个化合物到底能不能稳定存在？**——答案藏在**热力学凸包（convex hull）** 里。形成能为负**不等于**稳定：它还得比所有可能的分解产物组合更低；
3. **怎么从几万个候选里挑出值得合成的那几十个？**——把化学式变成特征向量，训练性质预测模型，再走一条**高通量虚拟筛选漏斗**。

最后我们会看到一个让人清醒的结果：一个随机划分下 $R^2 = 0.83$ 的带隙模型，一旦要求它外推到**从未见过的化学体系**，$R^2$ 会崩到 $0.19$。这是第二讲 Scaffold Split 教训在材料领域的翻版，而且更严重。

---

## 0. 环境准备与数据读取

In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull

warnings.filterwarnings("ignore")

from pymatgen.core import Structure, Lattice, Composition, Element, Species
from pymatgen.analysis.diffraction.xrd import XRDCalculator
from pymatgen.entries.computed_entries import ComputedEntry
from pymatgen.analysis.phase_diagram import PhaseDiagram

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, roc_auc_score, accuracy_score, confusion_matrix

plt.rcParams["figure.dpi"] = 110
np.random.seed(42)

def find_file(name):
    for path in (f"data/{name}", f"../data/{name}", name):
        if os.path.exists(path):
            return path
    try:
        from google.colab import files
        print(f"请上传 {name}")
        files.upload()
        return name
    except ImportError:
        raise FileNotFoundError(f"找不到 {name}，请确保它位于 data/ 文件夹或当前目录下")

print("本节用到的四个数据集：")
for f in ["bandgap_aflow.csv", "energy_per_atom_aflow.csv",
          "magpie_element_properties.csv", "perovskite_stability.csv"]:
    print(f"  {f:34s} {os.path.getsize(find_file(f)) / 1024:8.1f} KB")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---

## 1. 布拉格定律与模拟 XRD

**X 射线衍射（XRD）** 是材料实验室最常用的表征手段，没有之一。它的物理基础是**布拉格定律**：当波长 $\lambda$ 的 X 射线以掠射角 $\theta$ 入射到间距为 $d$ 的晶面族上，相邻晶面反射的光程差为 $2d\sin\theta$。只有当光程差等于波长整数倍时才相长干涉：

$$2 d_{hkl} \sin\theta = n\lambda$$

实验上我们扫描 $2\theta$，记录强度峰的位置。每个峰对应一族晶面 $(hkl)$，峰位由晶胞参数决定，**峰的强度则由晶胞内原子的排布决定**——这就是**结构因子（structure factor）**：

$$F_{hkl} = \sum_{j} f_j \, \exp\!\big[2\pi i (h x_j + k y_j + l z_j)\big]$$

求和跑遍晶胞内所有原子，$f_j$ 是第 $j$ 个原子的原子散射因子（大致正比于电子数 $Z$），$(x_j, y_j, z_j)$ 是它的分数坐标。衍射强度 $I_{hkl} \propto |F_{hkl}|^2$。

**当 $F_{hkl} = 0$ 时，这个峰就消失了——这叫系统消光（systematic absence）。** 消光规律是判断晶格类型（简单 / 体心 / 面心）最直接的证据。

pymatgen 的 `XRDCalculator` 把这一整套算得清清楚楚。下面先算最简单的 NaCl（Cu K$\alpha$ 辐射，$\lambda = 1.5406$ Å）：

In [ ]:
nacl = Structure.from_spacegroup(
    "Fm-3m", Lattice.cubic(5.64), ["Na", "Cl"], [[0, 0, 0], [0.5, 0.5, 0.5]]
)

xrd = XRDCalculator(wavelength="CuKa")          # lambda = 1.5406 A
pattern = xrd.get_pattern(nacl, two_theta_range=(20, 80))

print(f"{'2theta/deg':>11s} {'d/A':>8s} {'I_rel':>8s}   (hkl)")
for tt, inten, hkl_group, d in zip(pattern.x, pattern.y, pattern.hkls, pattern.d_hkls):
    hkl = hkl_group[0]["hkl"]
    mult = hkl_group[0]["multiplicity"]
    print(f"{tt:11.2f} {d:8.4f} {inten:8.1f}   {str(hkl):12s} 多重度={mult}")

# 用布拉格定律手工反算第一个峰的 d 值，验证 pymatgen 的结果
lam = xrd.wavelength          # Cu K-alpha 的加权平均波长，约 1.5418 A
tt0 = pattern.x[0]
d_manual = lam / (2 * np.sin(np.radians(tt0 / 2)))
print(f"\n手工用布拉格定律反算第一个峰: 2theta={tt0:.2f} deg -> d = {d_manual:.4f} A")
print(f"pymatgen 给出的 d_hkl        : {pattern.d_hkls[0]:.4f} A")

---

## 2. 系统消光：为什么金刚石硅没有 (200) 峰？

把 NaCl（岩盐，面心立方）和 Si（金刚石结构）的谱图并排放，就能看出结构因子的威力。两者都是面心立方点阵，但基元不同：

- **NaCl**：面心位置上交替放 Na$^+$ 和 Cl$^-$。面心立方的消光规律要求 $h, k, l$ **全奇或全偶**；而因为 Na 和 Cl 的电子数不同（11 vs 17），全奇的 $(111)$ 峰强度正比于 $|f_{\text{Na}} - f_{\text{Cl}}|^2$——**很弱但不为零**；
- **Si**：金刚石结构在每个面心格点上放**两个相同**的 Si，相对位移 $(\frac{1}{4},\frac{1}{4},\frac{1}{4})$。这额外引入一个条件：当 $h+k+l \equiv 2 \pmod 4$ 时 $F = 0$，于是 **$(200)$ 和 $(222)$ 峰彻底消失**。

下面直接验证：

In [ ]:
si = Structure.from_spacegroup("Fd-3m", Lattice.cubic(5.431), ["Si"], [[0, 0, 0]])
kcl = Structure.from_spacegroup(
    "Fm-3m", Lattice.cubic(6.29), ["K", "Cl"], [[0, 0, 0], [0.5, 0.5, 0.5]]
)

fig, axes = plt.subplots(3, 1, figsize=(10, 8.2), sharex=True)
for ax, (name, s, note) in zip(axes, [
    ("NaCl (rock salt)", nacl, "(111) 弱: f_Na != f_Cl"),
    ("KCl (rock salt)",  kcl,  "(111) 几乎消失: K+ 与 Cl- 等电子"),
    ("Si (diamond)",     si,   "(200)/(222) 系统消光"),
]):
    p = xrd.get_pattern(s, two_theta_range=(20, 80))
    ax.vlines(p.x, 0, p.y, color=C.navy, lw=2.0)
    for tt, inten, hg in zip(p.x, p.y, p.hkls):
        if inten > 4:
            ax.text(tt, inten + 3, "".join(str(abs(v)) for v in hg[0]["hkl"]),
                    ha="center", fontsize=7.5)
    ax.set_ylim(0, 118)
    ax.set_ylabel("I (rel.)")
    ax.set_title(f"{name}   —   {note}", fontsize=10, loc="left")
axes[-1].set_xlabel("2theta / degree  (Cu K-alpha)")
plt.show()

# 定量对比三者的 (111) 与 (200) 强度比
print(f"{'material':16s}{'I(111)':>9s}{'I(200)':>9s}{'I111/I200':>14s}")
for name, s in [("NaCl", nacl), ("KCl", kcl), ("Si", si)]:
    p = xrd.get_pattern(s, two_theta_range=(20, 80))
    peaks = {"".join(str(abs(v)) for v in hg[0]["hkl"]): i for hg, i in zip(p.hkls, p.y)}
    i111, i200 = peaks.get("111", 0.0), peaks.get("200", 0.0)
    ratio = f"{i111 / i200:.3f}" if i200 > 1e-6 else "无 200 峰"
    print(f"{name:16s}{i111:9.1f}{i200:9.1f}{ratio:>14s}")

三行输出把结构因子讲透了：

- **NaCl**：$I(111)/I(200) \approx 0.08$，微弱但存在；
- **KCl**：K$^+$ 和 Cl$^-$ 都是 18 个电子（等电子对），$f_{\text{K}^+} \approx f_{\text{Cl}^-}$，于是 $(111)$ 几乎完全熄灭——KCl 的粉末图看起来像一个晶胞参数只有一半的简单立方。这是晶体学教科书里的经典例题；
- **Si**：$(200)$ 峰**根本不在列表里**，因为金刚石结构的附加消光条件把它抹掉了。

> **📌 实验提示**：这正是 XRD 物相鉴定的底层逻辑——峰的**位置**告诉你晶胞多大，峰的**有无与强弱**告诉你原子怎么排。只看峰位是会误判的。

---

## 3. 真实任务：锐钛矿还是金红石？

现在做一件实验室每天都在做的事。$\text{TiO}_2$ 的两个主要多晶型——锐钛矿和金红石——组成完全相同，XRD 是区分它们的标准手段：

- **锐钛矿**的最强峰是 $(101)$，位于 $2\theta \approx 25.3^\circ$；
- **金红石**的最强峰是 $(110)$，位于 $2\theta \approx 27.4^\circ$。

这两个峰相距只有 $2^\circ$，却是判断光催化剂物相组成的金标准。下面我们用 `06a` 里的文献晶胞参数把两个相搭出来，算谱图，再模拟一个**两相混合样品**（就像商品 P25 催化剂那样）。

In [ ]:
anatase = Structure.from_spacegroup(
    "I4_1/amd", Lattice.tetragonal(3.785, 9.5196), ["Ti", "O"],
    [[0, 0, 0], [0, 0, 0.2081]]
)
rutile = Structure.from_spacegroup(
    "P4_2/mnm", Lattice.tetragonal(4.5937, 2.9587), ["Ti", "O"],
    [[0, 0, 0], [0.3053, 0.3053, 0]]
)

for name, s in [("anatase", anatase), ("rutile", rutile)]:
    p = xrd.get_pattern(s, two_theta_range=(20, 60))
    k = int(np.argmax(p.y))
    hkl = p.hkls[k][0]["hkl"]
    print(f"{name:9s} rho={s.density:.3f} g/cm3   最强峰 2theta={p.x[k]:.2f} deg  "
          f"(hkl)={hkl}  d={p.d_hkls[k]:.4f} A")

In [ ]:
def broaden(pattern, two_theta, fwhm=0.25):
    """把离散的 Bragg 峰用高斯展宽成连续谱线，模拟真实衍射仪的分辨率。"""
    sigma = fwhm / (2 * np.sqrt(2 * np.log(2)))
    y = np.zeros_like(two_theta)
    for pos, inten in zip(pattern.x, pattern.y):
        y += inten * np.exp(-0.5 * ((two_theta - pos) / sigma) ** 2)
    return y

tt_grid = np.linspace(20, 60, 2000)
p_ana = xrd.get_pattern(anatase, two_theta_range=(20, 60))
p_rut = xrd.get_pattern(rutile,  two_theta_range=(20, 60))
y_ana = broaden(p_ana, tt_grid)
y_rut = broaden(p_rut, tt_grid)

MIX = 0.80                       # 锐钛矿所占的强度分数
y_mix = MIX * y_ana + (1 - MIX) * y_rut

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True,
                         gridspec_kw={"height_ratios": [1.15, 1]})
axes[0].plot(tt_grid, y_ana, color=C.navy, lw=1.6, label="Anatase  I4_1/amd")
axes[0].plot(tt_grid, y_rut + 120, color=C.crimson, lw=1.6, label="Rutile  P4_2/mnm")
axes[0].axvline(25.28, color=C.navy, ls=":", lw=1.2)
axes[0].axvline(27.45, color=C.crimson, ls=":", lw=1.2)
axes[0].text(25.28, 243, "A(101)", ha="center", fontsize=8, color=C.navy)
axes[0].text(27.45, 243, "R(110)", ha="center", fontsize=8, color=C.crimson)
axes[0].set_ylabel("I (rel., 已上下偏移)")
axes[0].set_title("两个纯相的模拟 XRD 谱", fontsize=11, loc="left")
axes[0].legend(fontsize=9)

axes[1].plot(tt_grid, y_mix, color=C.gold, lw=1.8,
             label=f"混合物: {MIX:.0%} anatase + {1 - MIX:.0%} rutile")
axes[1].axvline(25.28, color=C.navy, ls=":", lw=1.2)
axes[1].axvline(27.45, color=C.crimson, ls=":", lw=1.2)
axes[1].set_xlabel("2theta / degree  (Cu K-alpha)")
axes[1].set_ylabel("I (rel.)")
axes[1].set_title("两相混合样品（类似商品 P25 光催化剂）", fontsize=11, loc="left")
axes[1].legend(fontsize=9)
plt.show()

In [ ]:
# 用 Spurr-Myers 公式从两个特征峰的强度比反算锐钛矿含量
#   W_A = 1 / (1 + 1.26 * I_R(110) / I_A(101))
def peak_height(y, grid, pos, window=0.6):
    m = np.abs(grid - pos) < window
    return y[m].max()

print(f"{'输入强度分数':>14s}{'I_A(101)':>12s}{'I_R(110)':>12s}{'Spurr-Myers W_A':>18s}")
for frac in [1.0, 0.8, 0.5, 0.2, 0.0]:
    y = frac * y_ana + (1 - frac) * y_rut
    iA = peak_height(y, tt_grid, 25.28)
    iR = peak_height(y, tt_grid, 27.45)
    w_a = 1.0 / (1.0 + 1.26 * iR / iA) if iA > 0 else 0.0
    print(f"{frac:14.0%}{iA:12.1f}{iR:12.1f}{w_a:18.1%}")

> **📌 为什么估计值和输入的分数对不上？** 因为 Spurr–Myers 公式里的常数 1.26 是把**强度比**换算成**质量比**的经验系数，而我们这里是按强度线性叠加的，两者口径不同。要真正做定量物相分析（QPA），必须用 Rietveld 全谱拟合，同时考虑标度因子、择优取向、吸收和峰形。这个示例的目的是让你看清**从谱图到物相含量这条链路长什么样**，以及经验公式的适用边界在哪。

### 🏋️ 练习 06b-1：晶格常数变化如何移动衍射峰

固溶体中掺杂会让晶胞膨胀或收缩（Vegard 定律），XRD 峰位随之移动。这是判断掺杂是否真的进入晶格的关键证据。

对金红石 $\text{TiO}_2$（`P4_2/mnm`，$a = 4.5937$ Å，$c = 2.9587$ Å）：

1. 让 $a$ 和 $c$ 同时按比例放大 $0\%, 1\%, 2\%, 3\%$（模拟掺入更大的阳离子），每次重新构建结构并计算 XRD；
2. 记录 $(110)$ 最强峰的 $2\theta$ 位置，画出“晶格膨胀率 vs 峰位”的曲线；
3. 用布拉格定律**手工推导**峰位移与晶格常数的关系：若 $d \to (1+\varepsilon)d$，证明 $\Delta(2\theta) \approx -2\varepsilon\tan\theta$（弧度制），并把你的数值结果与这个公式对比；
4. **讨论**：高角度峰（大 $2\theta$）还是低角度峰对晶格常数变化更敏感？这对精修晶胞参数时该用哪些峰有什么启示？

In [ ]:
# 在下方填写你的代码：


---

## 4. 热力学凸包：形成能为负 $\ne$ 稳定

这是材料信息学里**最容易被误解**的概念，必须讲透。

**形成能（formation energy）** $E_f$ 是化合物相对于**单质参考态**的能量（按每原子归一化）：

$$E_f(\text{Ti}_x\text{O}_y) = E(\text{Ti}_x\text{O}_y) - x'\,\mu_{\text{Ti}} - y'\,\mu_{\text{O}}$$

其中 $x', y'$ 是归一化后的原子分数。$E_f < 0$ 只说明这个化合物比“一堆单质放在一起”更稳定。

**但这远远不够。** 真正的问题是：它会不会分解成**别的化合物**？比如 $\text{Ti}_2\text{O}_3$ 的 $E_f = -2.96$ eV/atom，很负，可它仍然可能分解成 TiO 和 $\text{Ti}_4\text{O}_7$ 的混合物，只要右边的总能量更低。

**凸包（convex hull）** 就是这个判据的几何化：把所有化合物画在“组成 vs 形成能”图上，取这些点的**下凸包络**。

- 落在凸包**上**的相：$E_{\text{hull}} = 0$，**热力学稳定**，分解不划算；
- 落在凸包**上方**的相：$E_{\text{hull}} > 0$，**亚稳或不稳定**。$E_{\text{hull}}$ 的物理意义就是“分解能释放多少能量”。

经验判据：$E_{\text{hull}} < 25$ meV/atom（约等于室温 $k_BT$）通常认为可以合成；$> 100$ meV/atom 基本没戏。

我们用 **Ti–O 体系**来演示——它有一串著名的 **Magnéli 相** $\text{Ti}_n\text{O}_{2n-1}$，是检验凸包方法的绝佳例子。数据来自打包好的 `energy_per_atom_aflow.csv`（AFLOW 数据库的 DFT 总能）。

In [ ]:
ea = pd.read_csv(find_file("energy_per_atom_aflow.csv"))
TOKEN = re.compile(r"([A-Z][a-z]?)(\d*\.?\d*)")

def parse_formula(f):
    """把 'O2Ti1' 这样的化学式解析成 {'O': 2.0, 'Ti': 1.0}。"""
    d = {}
    for el, n in TOKEN.findall(f):
        if el:
            d[el] = d.get(el, 0.0) + (float(n) if n else 1.0)
    return d

ea["comp"] = ea.formula.map(parse_formula)
ea["n_el"] = ea.comp.map(len)

# 挑出单质条目，作为计算形成能的参考态
elemental = ea[ea.n_el == 1].copy()
elemental["element"] = elemental.comp.map(lambda c: next(iter(c)))

print("同一个元素往往有多个同素异形体条目，能量各不相同：")
for el in ["O", "N", "S", "C"]:
    g = elemental[elemental.element == el]
    print(f"  {el}: " + ",  ".join(f"{f}={v:.3f}" for f, v in
                                   zip(g.formula, g.energy_per_atom)))

看 **S** 那一行：有一条 `S8 = -23.559` eV/atom。硫的单质能量不可能是这个量级（同族条目都在 $-4.1$ 附近）——**这是公开数据集里的一条脏数据**。

这看起来只是一条坏记录，后果却极其严重：如果我们按“取同素异形体里能量最低的那个”来选参考态，这条 $-23.6$ 会被选中，于是**数据库里每一个含硫化合物的形成能都会错掉约 20 eV/atom**。一条脏数据污染上千条结果。

稳健的做法：对每个元素，先用**中位数**定位合理区间，剔除偏离超过 2 eV 的异常条目，再在剩下的里面取最小值。

In [ ]:
OUTLIER_TOL = 2.0        # eV/atom，偏离中位数超过这个值就判为异常

mu, flagged = {}, []
for el, g in elemental.groupby("element"):
    med = g.energy_per_atom.median()
    ok  = g[(g.energy_per_atom - med).abs() <= OUTLIER_TOL]
    bad = g[(g.energy_per_atom - med).abs() >  OUTLIER_TOL]
    for f, v in zip(bad.formula, bad.energy_per_atom):
        flagged.append({"element": el, "formula": f,
                        "energy": round(v, 3), "median": round(med, 3)})
    mu[el] = ok.energy_per_atom.min() if len(ok) else med

print(f"共 {len(mu)} 个元素有参考态；被判为异常并剔除的条目：")
print(pd.DataFrame(flagged).to_string(index=False))

def formation_energy(comp, e_per_atom):
    """每原子形成能：E_f = E - sum(x_i * mu_i)"""
    total = sum(comp.values())
    if not all(el in mu for el in comp):
        return np.nan
    return e_per_atom - sum(n / total * mu[el] for el, n in comp.items())

ea["E_f"] = [formation_energy(c, v) for c, v in zip(ea.comp, ea.energy_per_atom)]

print("\n与实验值对照（实验 dHf 来自标准热力学手册，单位 eV/atom）：")
CHECK = {"O2Ti1": -3.26, "Al2O3": -3.47, "O2Si1": -3.14,
         "Cl1Na1": -2.13, "O1Zn1": -1.81, "Cd1Te1": -0.49}
for f, exp in CHECK.items():
    row = ea[ea.formula == f]
    if len(row):
        print(f"  {Composition(f).reduced_formula:8s} DFT = {row.E_f.iloc[0]:+.3f}   "
              f"实验 = {exp:+.2f}")

In [ ]:
# ---- 取出 Ti-O 体系，手写凸包 ----
SYSTEM = {"Ti", "O"}
sub = ea[ea.comp.map(lambda c: len(c) > 0 and set(c) <= SYSTEM)].copy()
sub["x_O"] = sub.comp.map(lambda c: c.get("O", 0) / sum(c.values()))
sub["label"] = sub.comp.map(lambda c: Composition(
    "".join(f"{k}{v:g}" for k, v in c.items())).reduced_formula)
sub = sub.sort_values("x_O").reset_index(drop=True)
print(f"Ti-O 体系共 {len(sub)} 条记录（含同一化学式的不同多晶型）\n")

# 对每个组成只保留能量最低的多晶型，再求下凸包
best = sub.loc[sub.groupby("x_O").E_f.idxmin()].reset_index(drop=True)
pts = np.column_stack([best.x_O.values, best.E_f.values])

# 技巧：补一个远在上方的辅助点，ConvexHull 的下半部分就是我们要的
aug = np.vstack([pts, [0.5, pts[:, 1].max() + 10.0]])
hull = ConvexHull(aug)
hull_idx = sorted(i for i in hull.vertices if i < len(pts))
hull_pts = pts[hull_idx]

print("落在凸包上的稳定相：")
for i in hull_idx:
    print(f"  x_O = {best.x_O[i]:.4f}   {best.label[i]:8s}  E_f = {best.E_f[i]:+.4f} eV/atom")

def e_above_hull(x, ef, hull_pts):
    """线性插值出凸包在组成 x 处的高度，二者之差就是 E_hull。"""
    return ef - np.interp(x, hull_pts[:, 0], hull_pts[:, 1])

sub["E_hull"] = [e_above_hull(x, ef, hull_pts) for x, ef in zip(sub.x_O, sub.E_f)]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.4))

ax.scatter(sub.x_O, sub.E_f, s=45, color="#b0b0b0", zorder=2,
           label="全部计算条目（含亚稳多晶型）")
ax.plot(hull_pts[:, 0], hull_pts[:, 1], "-o", color=C.crimson, lw=2.0, ms=9,
        zorder=4, label="热力学凸包（稳定相）")

for i in hull_idx:
    ax.annotate(best.label[i], (best.x_O[i], best.E_f[i]),
                textcoords="offset points", xytext=(0, -17),
                ha="center", fontsize=9, color=C.crimson, fontweight="bold")

for lbl in ["Ti2O3", "Ti5O9", "Ti6O11"]:
    r = sub[sub.label == lbl]
    if len(r):
        r = r.loc[r.E_f.idxmin()]
        ax.annotate(f"{lbl}\n+{r.E_hull * 1000:.0f} meV",
                    (r.x_O, r.E_f), textcoords="offset points", xytext=(10, 12),
                    fontsize=8, color=C.navy,
                    arrowprops=dict(arrowstyle="->", color=C.navy, lw=0.8))

ax.axhline(0, color="#777777", lw=1.0, ls="--")
ax.set_xlabel("x(O)  =  O / (Ti + O)")
ax.set_ylabel("Formation energy  /  eV per atom")
ax.set_title("Ti-O 体系的热力学凸包（数据：AFLOW DFT 总能）")
ax.legend(fontsize=9, loc="lower left")
plt.show()

In [ ]:
# 同一件事用 pymatgen 的 PhaseDiagram 做一遍，并读出分解产物
entries = []
for comp, e in zip(sub.comp, sub.energy_per_atom):
    c = Composition("".join(f"{k}{v:g}" for k, v in comp.items()))
    entries.append(ComputedEntry(c, e * c.num_atoms))

pd_obj = PhaseDiagram(entries)
print("pymatgen 判定的稳定相:",
      sorted({e.composition.reduced_formula for e in pd_obj.stable_entries}))

print("\n几个亚稳相的分解路径：")
for target in ["Ti2O3", "Ti5O9", "Ti6O11"]:
    cands = [e for e in entries if e.composition.reduced_formula == target]
    if not cands:
        continue
    entry = min(cands, key=lambda e: e.energy_per_atom)
    decomp, ehull = pd_obj.get_decomp_and_e_above_hull(entry)
    route = "  +  ".join(f"{amt:.2f} {d.composition.reduced_formula}"
                         for d, amt in decomp.items())
    print(f"  {target:7s} E_hull = {ehull * 1000:6.1f} meV/atom   ->   {route}")

这张图和这几行输出把凸包讲完了：

- **稳定相**（红线上）是 Ti、$\text{Ti}_6\text{O}$、$\text{Ti}_3\text{O}$、$\text{Ti}_2\text{O}$、TiO、$\text{Ti}_4\text{O}_7$、$\text{TiO}_2$ 和 $\text{O}_2$。注意 $\text{TiO}_2$ 的 $E_f \approx -3.16$ eV/atom，与实验的 $-3.26$ 只差 0.1——DFT 对氧化物形成能的精度就在这个量级；
- **Magnéli 相**：$\text{Ti}_5\text{O}_9$ 只高出凸包约 7 meV/atom，$\text{Ti}_6\text{O}_{11}$ 约 13 meV/atom——**都远小于室温 $k_BT \approx 25$ meV**。所以它们在实验上确实能稳定存在（Magnéli 相正是商业化的 Ebonex 导电陶瓷）；
- **$\text{Ti}_2\text{O}_3$** 高出约 32 meV/atom，仍属“可以合成的亚稳相”；
- 同一个化学式会有**多个**点（不同多晶型）：TiO 有两个条目，一个在凸包上，另一个高出约 360 meV/atom。**$E_{\text{hull}}$ 永远是相对于同组成最低能多晶型来说的**。

> **📌 凸包的三条实战要点**
> 1. $E_f < 0$ 只是必要条件，$E_{\text{hull}} = 0$ 才是稳定；
> 2. 凸包的形状取决于你放了哪些相进去——**漏掉一个竞争相，就会把一个不稳定的东西误判成稳定**。这是高通量筛选最常见的假阳性来源；
> 3. 凸包是 0 K、不含熵的判据。高温下构型熵可能让 $E_{\text{hull}} \approx 50$ meV/atom 的相反而稳定（固溶体尤其如此）。

### 🏋️ 练习 06b-2：另一个二元体系的凸包

把上面的流程封装成一个函数 `build_hull(system)`（`system` 是元素集合，如 `{"Al", "O"}`），对下面三个体系各做一次凸包分析：

- $\text{Al}-\text{O}$（只有一个著名化合物 $\text{Al}_2\text{O}_3$）
- $\text{Li}-\text{O}$（$\text{Li}_2\text{O}$ 与过氧化物 $\text{Li}_2\text{O}_2$，锂空气电池的关键相）
- $\text{Fe}-\text{O}$（FeO / $\text{Fe}_3\text{O}_4$ / $\text{Fe}_2\text{O}_3$，经典的多价氧化物体系）

对每个体系：
1. 打印落在凸包上的稳定相及其 $E_f$；
2. 画出凸包图；
3. 找出 $E_{\text{hull}}$ 最小的**非**稳定相，用 `PhaseDiagram.get_decomp_and_e_above_hull` 给出它的分解路径；
4. **讨论**：Fe–O 体系里哪些相在凸包上？与你在相图教科书里学到的（方铁矿 / 磁铁矿 / 赤铁矿）是否一致？如果不一致，可能是什么原因？

> **提示**：Fe 是磁性元素，不加 Hubbard $U$ 的普通 DFT 对铁氧化物形成能误差较大——这本身就是答案的一部分。

In [ ]:
# 在下方填写你的代码：


---

## 5. 钙钛矿稳定性：Goldschmidt 容差因子

凸包要靠 DFT 算，很贵。有没有**不用算就能预判稳定性**的经验规则？钙钛矿领域有一个用了快一百年的答案：**Goldschmidt 容差因子（1926）**

$$t = \frac{r_A + r_X}{\sqrt{2}\,(r_B + r_X)}$$

它的几何意义很直白：在理想立方钙钛矿 $ABX_3$ 里，A 位阳离子要恰好填进由 $BX_6$ 八面体围成的立方空腔。$t = 1$ 代表完美匹配。经验规律是：

| $t$ 范围 | 结构倾向 |
|---|---|
| $t < 0.8$ | A 位太小，塌成钛铁矿等非钙钛矿结构 |
| $0.8 < t < 1.0$ | **钙钛矿稳定窗口**（$t$ 略小于 1 时八面体倾转，变成正交相） |
| $t > 1.0$ | A 位太大，倾向六方钙钛矿或分层结构 |

下面我们用 1,929 条**已发表的 DFT 计算数据**（含 $E_{\text{hull}}$）来检验这条 1926 年的规则还灵不灵。离子半径用 pymatgen 内置的 **Shannon 半径**——注意它依赖**氧化态和配位数**，这正是用好它的关键。

In [ ]:
R_O = 1.40           # Shannon 半径：O2- 在六配位下为 1.40 A
A_CN = ("XII", "IX", "VIII", "VI")      # A 位优先按 12 配位查，查不到就退到低配位
B_CN = ("VI",)                           # B 位固定六配位

def shannon_radius(el, oxi, cns):
    for cn in cns:
        try:
            r = Species(el, oxi).get_shannon_radius(cn)
            if r:
                return float(r)
        except Exception:
            pass
    try:
        return float(Species(el, oxi).ionic_radius)
    except Exception:
        return None

perov = pd.read_csv(find_file("perovskite_stability.csv"))
print(f"钙钛矿稳定性数据集: {len(perov)} 条")
print(perov.head(3).to_string(index=False))

rows = []
for f, eh in zip(perov.formula, perov.e_above_hull_meV):
    d = Composition(f).reduced_composition.get_el_amt_dict()
    cations = {k: v for k, v in d.items() if k != "O"}
    # 只取干净的 ABO3（两种阳离子、1:1:3），混合占位的留给练习
    if len(cations) != 2 or abs(d.get("O", 0) - 3) > 1e-6:
        continue
    if any(abs(v - 1) > 1e-6 for v in cations.values()):
        continue
    # A 位 = 电负性更小（更正电性）的那个阳离子
    A, B = sorted(cations, key=lambda e: Element(e).X)
    for qA, qB in [(2, 4), (3, 3), (1, 5)]:          # 常见的三种电荷配对
        rA = shannon_radius(A, qA, A_CN)
        rB = shannon_radius(B, qB, B_CN)
        if rA and rB:
            rows.append({"formula": Composition(f).reduced_formula,
                         "A": A, "qA": qA, "B": B, "qB": qB,
                         "r_A": rA, "r_B": rB,
                         "t": (rA + R_O) / (np.sqrt(2) * (rB + R_O)),
                         "E_hull_meV": eh})
            break

tol = pd.DataFrame(rows)
print(f"\n可算容差因子的干净 ABO3: {len(tol)} 条")
print("E_hull 最低的 5 个（即最稳定的）：")
print(tol.nsmallest(5, "E_hull_meV").round(3).to_string(index=False))

In [ ]:
STABLE_CUT = 50.0          # meV/atom，常用的"可合成"阈值

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

ax = axes[0]
colors = np.where(tol.E_hull_meV < STABLE_CUT, C.navy, "#c7c7c7")
ax.scatter(tol.t, tol.E_hull_meV, c=colors, s=46, edgecolor="white", linewidth=0.5)
ax.axvspan(0.8, 1.0, color=C.gold, alpha=0.18, label="Goldschmidt 窗口 0.8 < t < 1.0")
ax.axhline(STABLE_CUT, color=C.crimson, ls="--", lw=1.3,
           label=f"E_hull = {STABLE_CUT:.0f} meV/atom")
ax.set_xlabel("Goldschmidt tolerance factor  t")
ax.set_ylabel("E above hull  /  meV per atom")
ax.set_title("容差因子 vs DFT 稳定性", fontsize=11)
ax.legend(fontsize=8)

stats = []
for lo, hi in [(0.0, 0.8), (0.8, 0.9), (0.9, 1.0), (1.0, 1.2)]:
    s = tol[(tol.t >= lo) & (tol.t < hi)]
    if len(s):
        stats.append({"t_range": f"[{lo}, {hi})", "n": len(s),
                      "median_E_hull": s.E_hull_meV.median(),
                      "frac_stable": (s.E_hull_meV < STABLE_CUT).mean()})
stats = pd.DataFrame(stats)

ax = axes[1]
bars = ax.bar(stats.t_range, stats.frac_stable,
              color=[C.gold if r.startswith(("[0.8", "[0.9")) else C.navy
                     for r in stats.t_range], width=0.6)
for b, n, m in zip(bars, stats.n, stats.median_E_hull):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.025,
            f"n={n}\n中位数 {m:.0f}", ha="center", fontsize=8)
ax.set_ylim(0, 1.0)
ax.set_xlabel("tolerance factor t")
ax.set_ylabel(f"稳定比例 (E_hull < {STABLE_CUT:.0f} meV)")
ax.set_title("分箱统计：窗口内外的稳定率", fontsize=11)
plt.show()

print(stats.round(3).to_string(index=False))
inside  = tol[(tol.t >= 0.8) & (tol.t < 1.0)]
outside = tol[(tol.t < 0.8) | (tol.t >= 1.0)]
print(f"\n窗口内 E_hull 中位数 = {inside.E_hull_meV.median():.1f} meV/atom (n={len(inside)})")
print(f"窗口外 E_hull 中位数 = {outside.E_hull_meV.median():.1f} meV/atom (n={len(outside)})")
print(f"而 |t-1| 与 E_hull 的线性相关系数只有 r = "
      f"{tol.E_hull_meV.corr((tol.t - 1).abs()):.3f}")

这组结果非常值得玩味：

- **分箱来看，Goldschmidt 规则完全成立**：窗口内（$0.8 < t < 1.0$）的 $E_{\text{hull}}$ 中位数约 34 meV/atom，窗口外约 91 meV——差了快 3 倍；窗口内有六到七成的组成可以合成，$t > 1.0$ 的那一箱只剩三成；
- **但线性相关系数只有 0.07**。

这不矛盾，而是在告诉我们一件重要的事：**$t$ 是一个“窗口型”判据，不是一个“排序型”预测器**。它能帮你把明显不可能的组成筛掉，却无法在窗口内部告诉你谁更稳定——因为窗口内的稳定性由电子结构（Jahn–Teller 畸变、$d$ 电子组态、共价性）决定，而这些信息根本不在“三个离子半径”里。

**这恰恰是机器学习该上场的地方**：用更丰富的组成特征去捕捉单个几何描述符捕捉不到的东西。

---

## 6. 组成描述符（CBFV）：把化学式变成特征向量

分子有 SMILES 和指纹，晶体结构有 pymatgen。但如果我们**只知道化学式**（高通量筛选的早期阶段往往如此，结构还没算出来），该怎么构造特征？

答案是 **CBFV（Composition-Based Feature Vector，组成特征向量）**。思路很朴素：

1. 先准备一张**元素性质表**——每个元素的原子序数、电负性、共价半径、熔点、价电子数……本讲用的是经典的 **Magpie** 属性集（Ward et al., *npj Comput. Mater.* **2**, 16028, 2016），共 22 个元素属性；
2. 对一个化学式（比如 $\text{Ti}_1\text{O}_2$），按化学计量数算出每个属性的**统计量**：加权平均、加权绝对偏差、最小值、最大值、极差；
3. $22 \times 5 = 110$ 维向量，就是这个化学式的“指纹”。

这正是**分子描述符思想在无机材料上的平移**：分子用原子贡献加和得到 $\log P$，晶体用元素属性加权得到组成特征。

In [ ]:
magpie = pd.read_csv(find_file("magpie_element_properties.csv")).set_index("element")
PROPS = list(magpie.columns)
LOOKUP = magpie.to_dict("index")
print(f"Magpie 元素属性表: {magpie.shape[0]} 个元素 x {len(PROPS)} 个属性")
print("属性列表:", PROPS)
print()
print(magpie.loc[["Ti", "O", "Si", "Cu"]].round(2).to_string())

STATS = ["mean", "avg_dev", "min", "max", "range"]
FEATURE_NAMES = [f"{s}_{p}" for s in STATS for p in PROPS]

def composition_features(comp):
    """把 {'Ti':1,'O':2} 变成 110 维 CBFV 向量；有元素查不到时返回 None。"""
    els = list(comp)
    if not els or not all(e in LOOKUP for e in els):
        return None
    w = np.array([comp[e] for e in els], dtype=float)
    w = w / w.sum()
    M_ = np.array([[LOOKUP[e][p] for p in PROPS] for e in els], dtype=float)
    mean = (w[:, None] * M_).sum(0)
    avg_dev = (w[:, None] * np.abs(M_ - mean)).sum(0)
    return np.concatenate([mean, avg_dev, M_.min(0), M_.max(0), M_.max(0) - M_.min(0)])

demo = composition_features(parse_formula("O2Ti1"))
print(f"\nTiO2 的 CBFV 维度 = {len(demo)}")
for k in [0, 7, 22, 44, 66, 88]:
    print(f"  {FEATURE_NAMES[k]:28s} = {demo[k]:10.3f}")

---

## 7. 带隙预测：回归与“是不是金属”分类

数据集是 `bandgap_aflow.csv`：27,767 个无机化合物的 DFT（PBE 泛函）带隙，来自 AFLOW 数据库，经 Wang 等人的 *Machine Learning for Materials Scientists: An Introductory Guide toward Best Practices*（*Chem. Mater.* **32**, 4954, 2020）整理去重。

> **⚠️ 一个必须知道的物理限制**：PBE 泛函**系统性地低估带隙**，通常只有实验值的 $50\sim70\%$。所以这里训练出来的模型学的是“PBE 带隙”，不是“实验带隙”。把它的输出直接当实验值用，是材料机器学习论文里非常常见的错误。

In [ ]:
bg = pd.read_csv(find_file("bandgap_aflow.csv"))
bg["comp"] = bg.formula.map(parse_formula)
bg["n_el"] = bg.comp.map(len)
bg["elements"] = bg.comp.map(lambda c: frozenset(c))

feats, keep = [], []
for i, c in enumerate(bg.comp):
    v = composition_features(c)
    if v is not None:
        feats.append(v)
        keep.append(i)

X = np.asarray(feats)
bg = bg.iloc[keep].reset_index(drop=True)
y = bg.band_gap.values

print(f"特征矩阵: {X.shape[0]} 个化合物 x {X.shape[1]} 维 CBFV")
print(f"带隙范围: {y.min():.2f} ~ {y.max():.2f} eV")
print(f"零带隙（金属）比例: {(y < 0.01).mean():.1%}")
print(f"\n元素数分布:\n{bg.n_el.value_counts().sort_index().to_string()}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(y, bins=60, color=C.navy)
axes[0].set_yscale("log")
axes[0].set_xlabel("PBE band gap / eV")
axes[0].set_ylabel("Count (log)")
axes[0].set_title("带隙分布：将近一半是金属", fontsize=10)
axes[1].hist(y[y > 0.01], bins=60, color=C.gold)
axes[1].set_xlabel("PBE band gap / eV")
axes[1].set_ylabel("Count")
axes[1].set_title("只看非金属部分", fontsize=10)
plt.show()

In [ ]:
# ① 回归：预测带隙数值
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
reg = RandomForestRegressor(n_estimators=150, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
pred = reg.predict(X_te)
r2_random = r2_score(y_te, pred)
print(f"[回归] 随机划分  R2 = {r2_random:.4f}   "
      f"MAE = {np.abs(pred - y_te).mean():.3f} eV   "
      f"RMSE = {np.sqrt(((pred - y_te) ** 2).mean()):.3f} eV")

# ② 分类：先判断"是不是金属"（工业筛选里往往这一步就够了）
y_cls = (y > 0.01).astype(int)
Xc_tr, Xc_te, c_tr, c_te = train_test_split(X, y_cls, test_size=0.2, random_state=42)
clf = RandomForestClassifier(n_estimators=150, random_state=0, n_jobs=-1).fit(Xc_tr, c_tr)
proba = clf.predict_proba(Xc_te)[:, 1]
print(f"[分类] 金属 vs 非金属  准确率 = {accuracy_score(c_te, clf.predict(Xc_te)):.4f}   "
      f"ROC-AUC = {roc_auc_score(c_te, proba):.4f}")
print("\n混淆矩阵:")
print(pd.DataFrame(confusion_matrix(c_te, clf.predict(Xc_te)),
                   index=["真实:金属", "真实:非金属"],
                   columns=["预测:金属", "预测:非金属"]).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))
axes[0].scatter(y_te, pred, s=7, alpha=0.3, color=C.navy, edgecolor="none")
axes[0].plot([0, 12], [0, 12], "--", color="#888888", lw=1.2)
axes[0].set_xlim(0, 12)
axes[0].set_ylim(0, 12)
axes[0].set_xlabel("DFT (PBE) band gap / eV")
axes[0].set_ylabel("Predicted band gap / eV")
axes[0].set_title(f"随机划分: R2 = {r2_random:.3f}", fontsize=11)

imp = pd.Series(reg.feature_importances_, index=FEATURE_NAMES).nlargest(12).iloc[::-1]
axes[1].barh(imp.index, imp.values, color=C.gold)
axes[1].set_xlabel("Random-forest feature importance")
axes[1].set_title("最重要的 12 个组成特征", fontsize=11)
axes[1].tick_params(axis="y", labelsize=7.5)
plt.show()

特征重要性里排在最前面的基本都是 **`GSbandgap`（元素单质的基态带隙）** 和**电负性**相关的统计量——完全符合化学直觉：由宽带隙元素（O、F、N）构成的化合物更可能是绝缘体，而电负性差大意味着离子性强、带隙宽。模型没有被喂任何物理公式，却自己找到了这条规律。

---

## 8. 外推悬崖：随机划分给出的 $R^2$ 是假的

$R^2 = 0.83$ 看起来很漂亮。但请想一想**高通量筛选真正的使用场景**：

> 我们训练模型，是为了用它预测**还没有人算过、甚至还没有人做过**的新化合物。

随机划分下，测试集里的某个钛酸盐旁边，训练集里往往躺着一整排化学上极其相似的钛酸盐——模型其实是在做**内插**。这和第二讲 `02b` 里随机划分 vs Scaffold Split 的问题一模一样，只不过在材料领域更隐蔽：分子至少有骨架可以聚类，而化学式连骨架都没有。

真正诚实的检验是**留出整类化学（leave-one-chemistry-out）**：把所有含某个元素的化合物整体挪到测试集，看模型能不能外推过去。

In [ ]:
def holdout_score(mask_test, name):
    tr = ~mask_test
    m = RandomForestRegressor(n_estimators=150, random_state=0, n_jobs=-1).fit(X[tr], y[tr])
    s = r2_score(y[mask_test], m.predict(X[mask_test]))
    print(f"  {name:34s} n_test = {mask_test.sum():5d}   R2 = {s:6.3f}")
    return {"split": name, "n_test": int(mask_test.sum()), "r2": s}

results = []
rng_mask = np.zeros(len(bg), bool)
_, te_idx = train_test_split(np.arange(len(bg)), test_size=0.2, random_state=42)
rng_mask[te_idx] = True

print("基线——随机划分：")
results.append(holdout_score(rng_mask, "random 20% split"))

print("\n留出整类化学（真正的外推）：")
for el in ["O", "S", "N", "F"]:
    results.append(holdout_score(
        bg.elements.map(lambda s: el in s).values, f"hold out all {el}-containing"))

LANTHANIDES = set("La Ce Pr Nd Pm Sm Eu Gd Tb Dy Ho Er Tm Yb Lu".split())
results.append(holdout_score(
    bg.elements.map(lambda s: bool(s & LANTHANIDES)).values,
    "hold out all lanthanide-containing"))

res = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(9, 4.4))
bars = ax.barh(res.split, res.r2, color=[C.navy] + [C.crimson] * (len(res) - 1))
for b, v, n in zip(bars, res.r2, res.n_test):
    ax.text(max(v, 0) + 0.015, b.get_y() + b.get_height() / 2,
            f"{v:.3f}  (n={n})", va="center", fontsize=8.5)
ax.invert_yaxis()
ax.set_xlim(0, 1.0)
ax.set_xlabel("Test R2")
ax.set_title("随机划分 vs 留出整类化学：外推能力的真相", fontsize=11)
ax.tick_params(axis="y", labelsize=8.5)
plt.show()

**这就是材料机器学习最重要的一张图。**

- 随机划分：$R^2 \approx 0.83$；
- 留出全部**氟化物**：$R^2 \approx 0.19$；
- 留出全部**氧化物**：$R^2 \approx 0.32$。

模型在没见过氟化物的情况下，对氟化物的带隙几乎一无所知——而这正是“用模型发现全新材料体系”时要面对的处境。

原因不难理解：CBFV 本质上是**元素属性的加权统计量**。如果训练集里从来没出现过 F，那么“F 的 `GSbandgap` 很大、电负性极高”这些信息虽然写在元素表里，模型却从没学过**这些数值该如何映射到带隙**。统计模型只能内插，不能凭空获得新的物理。

> **📌 报告模型性能时的三条底线**
> 1. 必须说清**划分方式**。只报随机划分的 $R^2$，等于什么也没说；
> 2. 对“用于发现新体系”的模型，**留出整类化学**才是诚实的评估；
> 3. 任何高通量筛选的结果都必须检查**适用域**——候选物的化学体系是否在训练集的覆盖范围内。这和第四讲 `04a` 里高斯过程给出的 $\sigma(x)$、第二讲的 Tanimoto 适用域检查是同一件事。

---

## 9. 高通量虚拟筛选漏斗

最后，把所有东西串成一条真实的流水线。任务：**从 27,767 个化合物里，筛出适合做太阳能电池吸收层的候选**。

筛选条件（每一条都有明确的物理依据）：

| 关卡 | 判据 | 物理依据 |
|---|---|---|
| ① 带隙窗口 | $1.1 \le E_g \le 1.7$ eV | Shockley–Queisser 极限在 1.34 eV 附近取最大效率 |
| ② 元素安全性 | 排除 Pb/Cd/Hg/As/Tl/Be 等 | 毒性与法规限制（RoHS） |
| ③ 资源可得性 | 排除贵金属与稀缺元素 | 太阳能电池需要太瓦级产能 |
| ④ 可合成性 | 元素数 $\le 3$ | 三元以上的相区控制极难 |
| ⑤ 热力学 | $E_f < -0.2$ eV/atom | 形成能为负，至少不会自发分解回单质 |

我们先**原封不动**地照着教科书上的带隙窗口跑一遍，然后用已知材料做健全性检验。

In [ ]:
TOXIC       = set("Pb Cd Hg As Tl Be Cr Ni Sb".split())
RADIOACTIVE = set("Tc Pm Po At Rn Fr Ra Ac Th Pa U Np Pu".split())
NOBLE       = set("He Ne Ar Kr Xe".split())
SCARCE      = set("Au Pt Ir Os Rh Ru Pd Re Sc Hf Ta".split())

pool = bg.merge(ea[["formula", "E_f"]], on="formula", how="left")
pool["key"] = pool.formula.map(lambda f: Composition(f).reduced_composition)

def run_funnel(gap_lo, gap_hi, ef_cut=-0.2, verbose=True):
    """跑一遍五关漏斗，返回 (入围表, 每关剩余数)。"""
    stages = [{"stage": "起始候选库", "n": len(pool)}]
    s = pool
    def gate(df, label):
        stages.append({"stage": label, "n": len(df)})
        if verbose:
            print(f"{label:38s} -> {len(df):6d}")
        return df
    if verbose:
        print(f"{'起始候选库':38s} -> {len(pool):6d}")
    s = gate(s[s.band_gap.between(gap_lo, gap_hi)], f"① 带隙 {gap_lo}-{gap_hi} eV")
    s = gate(s[~s.elements.map(lambda e: bool(e & (TOXIC | RADIOACTIVE | NOBLE)))],
             "② 排除有毒/放射性/惰性元素")
    s = gate(s[~s.elements.map(lambda e: bool(e & SCARCE))], "③ 排除贵金属与稀缺元素")
    s = gate(s[s.n_el <= 3], "④ 元素数 <= 3（可合成性）")
    s = gate(s[s.E_f < ef_cut], f"⑤ 形成能 < {ef_cut} eV/atom")
    return s.reset_index(drop=True), pd.DataFrame(stages)

print("=== 第一次尝试：直接套用实验上的最佳带隙窗口 1.1-1.7 eV ===")
shortlist_v1, funnel_v1 = run_funnel(1.1, 1.7)

show = shortlist_v1.nsmallest(12, "E_f").copy()
show["pretty"] = show.formula.map(lambda f: Composition(f).reduced_formula)
print(f"\n入围 {len(shortlist_v1)} 个，形成能最负的 12 个：\n")
print(show[["pretty", "band_gap", "E_f", "n_el"]]
      .rename(columns={"pretty": "formula", "band_gap": "Eg_eV", "E_f": "Ef_eV_atom"})
      .round(3).to_string(index=False))

In [ ]:
KNOWN_PV = ["CuInSe2", "CuGaSe2", "CuInS2", "AgInSe2", "CdTe", "InP", "GaAs"]
pool_keys = set(pool.key)
short_keys_v1 = set(shortlist_v1.key)

print("健全性检验——已知的光伏吸收层材料有没有被筛出来：")
print(f"{'material':12s}{'在候选库中':>12s}{'通过筛选':>12s}")
for m in KNOWN_PV:
    k = Composition(m).reduced_composition
    print(f"{m:12s}{str(k in pool_keys):>12s}{str(k in short_keys_v1):>12s}")

**健全性检验全军覆没。** 七个已知的光伏吸收层材料全都在候选库里，却**没有一个**通过筛选。

再看入围名单：清一色的**氟化物和氧化物**（$\text{CsEuF}_3$、$\text{LiYbF}_4$、$\text{VF}_3$……）。任何一个做过太阳能电池的人都知道，氟化物绝不可能是好的吸收层——它们离子性太强、载流子迁移率极低。

**这个筛选流程坏掉了。** 但它坏得很有价值，因为原因恰恰是我们在第 7 节开头强调过、却没有在筛选时兑现的那条警告：**PBE 泛函系统性低估带隙**。

我们来直接验证这个诊断——把已知光伏材料的 PBE 带隙打印出来看看：

In [ ]:
print(f"{'material':12s}{'PBE 带隙':>12s}{'实验带隙':>12s}{'PBE/实验':>12s}")
EXPERIMENTAL = {"CuInSe2": 1.04, "CuGaSe2": 1.68, "CuInS2": 1.53,
                "AgInSe2": 1.24, "CdTe": 1.49, "InP": 1.34, "GaAs": 1.42}
for m, eg_exp in EXPERIMENTAL.items():
    k = Composition(m).reduced_composition
    r = pool[pool.key == k]
    if len(r):
        eg_pbe = r.band_gap.iloc[0]
        ratio = f"{eg_pbe / eg_exp:.2f}" if eg_exp else "-"
        print(f"{m:12s}{eg_pbe:12.3f}{eg_exp:12.2f}{ratio:>12s}")

诊断确认：这些材料的 PBE 带隙全都落在 **$0.3 \sim 0.7$ eV**，只有实验值的三到五成。$\text{CuInSe}_2$ 更极端——PBE 直接把它算成了**金属**（$E_g = 0$），这是文献里有名的一个 DFT 失败案例。

所以，**在 PBE 数据上做筛选，就必须用 PBE 的尺子**。既然目标是实验带隙 $1.1 \sim 1.7$ eV，而 PBE 大约只给出实验值的 $30\% \sim 50\%$，那么对应的 PBE 窗口应该下移到大约 $0.3 \sim 1.0$ eV。

> 更严谨的做法是先用一批”PBE 值 + 实验值”配对数据拟合一个校正关系（文献上常用线性或分段线性校正），再把阈值映射过去。这里我们直接用上面这张表读出的经验比例。

下面用修正后的窗口重跑一遍：

In [ ]:
print("=== 第二次尝试：把带隙窗口换成 PBE 尺度下的 0.3-1.0 eV ===")
shortlist, funnel_v2 = run_funnel(0.3, 1.0)

short_keys = set(shortlist.key)
print("\n健全性检验（修正窗口后）：")
print(f"{'material':12s}{'通过筛选':>10s}   备注")
for m in KNOWN_PV:
    k = Composition(m).reduced_composition
    r = pool[pool.key == k]
    ok = k in short_keys
    note = ""
    if not ok and len(r):
        row = r.iloc[0]
        if not (0.3 <= row.band_gap <= 1.0):
            note = f"卡在关卡①: PBE Eg={row.band_gap:.3f}"
        elif row.elements & (TOXIC | RADIOACTIVE | NOBLE):
            note = f"卡在关卡②: 含 {sorted(row.elements & TOXIC)}"
        elif row.elements & SCARCE:
            note = f"卡在关卡③: 含稀缺元素"
        elif row.n_el > 3:
            note = "卡在关卡④: 元素数 > 3"
        elif not (row.E_f < -0.2):
            note = f"卡在关卡⑤: Ef={row.E_f:+.3f}"
    print(f"{m:12s}{str(ok):>10s}   {note}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8))

ax = axes[0]
w = 0.38
idx = np.arange(len(funnel_v2))
ax.barh(idx - w / 2, funnel_v1.n, height=w, color="#c7c7c7", label="窗口 1.1-1.7 eV（错误）")
ax.barh(idx + w / 2, funnel_v2.n, height=w, color=C.gold, label="窗口 0.3-1.0 eV（PBE 修正）")
for i, (n1, n2) in enumerate(zip(funnel_v1.n, funnel_v2.n)):
    ax.text(n1 * 1.1, i - w / 2, f"{n1:,}", va="center", fontsize=7.5, color="#666666")
    ax.text(n2 * 1.1, i + w / 2, f"{n2:,}", va="center", fontsize=7.5)
ax.set_yticks(idx)
ax.set_yticklabels(funnel_v2.stage, fontsize=8.5)
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_xlim(right=funnel_v2.n.max() * 6)
ax.set_xlabel("剩余候选数（对数坐标）")
ax.set_title("高通量虚拟筛选漏斗", fontsize=11)
ax.legend(fontsize=8, loc="lower right")

ax = axes[1]
ax.scatter(pool.band_gap, pool.E_f, s=4, alpha=0.10, color="#b0b0b0",
           edgecolor="none", label=f"全库 ({len(pool):,})")
ax.scatter(shortlist.band_gap, shortlist.E_f, s=22, color=C.crimson,
           edgecolor="white", linewidth=0.3, label=f"入围 ({len(shortlist)})")
for m in ["CuGaSe2", "CuInS2", "AgInSe2"]:
    r = pool[pool.key == Composition(m).reduced_composition]
    if len(r):
        ax.scatter(r.band_gap, r.E_f, s=120, marker="*", color=C.navy, zorder=5)
        ax.annotate(m, (r.band_gap.iloc[0], r.E_f.iloc[0]),
                    textcoords="offset points", xytext=(6, 6), fontsize=8, color=C.navy)
ax.axvspan(0.3, 1.0, color=C.gold, alpha=0.18)
ax.axhline(-0.2, color=C.navy, ls="--", lw=1.1)
ax.set_xlim(0, 4)
ax.set_ylim(-4.5, 1.0)
ax.set_xlabel("PBE band gap / eV")
ax.set_ylabel("Formation energy / eV per atom")
ax.set_title("修正窗口后：已知光伏材料落在入围区内", fontsize=11)
ax.legend(fontsize=8.5, loc="lower right")
plt.show()

修正之后，$\text{CuGaSe}_2$、$\text{CuInS}_2$、$\text{AgInSe}_2$、$\text{InP}$ 这些真正被工业化或被深入研究过的吸收层材料都回到了名单里；$\text{CdTe}$ 和 $\text{GaAs}$ 落选了。$	ext{CdTe}$ 是被**第 ② 关的毒性判据**（Cd）挡住的——这是判据在正确地起作用，不是 bug；而 $	ext{GaAs}$ 的 PBE 带隙是 0.295 eV，**差 0.005 eV 就卡在窗口下沿**，这是硬划阈值的代价。最后 $	ext{CuInSe}_2$ 更干脆——PBE 直接把它算成了金属（$E_g = 0$），任何基于这份数据的筛选都救不回它。

**这一整节的方法论价值，远大于那张候选名单本身**：

> **📌 高通量筛选的第一条铁律：先让流程筛出你已经知道的答案，再去相信它筛出的新答案。**

如果我们没做健全性检验，就会得到一份全是氟化物的”光伏候选材料”名单，而且每个数字看起来都人畜无害。筛选流程的失败往往不会报错，它只会安静地给你一份错误的名单。

> **这条流水线剩下的局限（都必须在论文里写清楚）**：
> 1. 我们用经验比例硬调了带隙窗口。更严谨的做法是拟合 PBE→实验的校正关系，或者直接用 HSE / GW 级别的数据；
> 2. 我们用的是 $E_f$ 而不是 $E_{\text{hull}}$——**形成能为负不等于稳定**（第 4 节的教训）。严格做法是对每个候选构建它所在化学体系的凸包；
> 3. 完全没有考虑载流子迁移率、缺陷容限、吸收系数、能带对齐——而这些往往才是真正的杀手；
> 4. 筛选用到的带隙是数据库里**已经算好的**。要筛**数据库里没有的**新组成，就得用第 7 节的模型去预测——这时第 8 节那个“外推悬崖”就会直接决定筛选结果可不可信。

### 🏋️ 练习 06b-3：设计你自己的筛选流程——透明导电氧化物（TCO）

透明导电氧化物（如 ITO）是所有显示器和太阳能电池都要用的电极材料。它的要求和光伏吸收层**正好相反**：

| 关卡 | 判据 |
|---|---|
| ① 必须含氧 | 化学式里有 O |
| ② 透明 | 带隙 $> 3.0$ eV（可见光不吸收） |
| ③ 结构简单易掺杂 | `n_el <= 3` |
| ④ 热力学可行 | $E_f < -1.0$ eV/atom |
| ⑤ 元素安全可得 | 排除有毒、放射性与贵金属元素 |

1. 照着第 9 节的 `run_funnel()` 写法实现这条漏斗，打印每一关剩下多少候选；
2. **先做健全性检验**（吸取本节的教训，不要留到最后）：$\text{In}_2\text{O}_3$、$\text{SnO}_2$、$\text{ZnO}$、$\text{Ga}_2\text{O}_3$ 这些已知的 TCO 母体材料，有几个通过了筛选？没通过的卡在哪一关？
3. 把这四个材料的 PBE 带隙打印出来，与它们的实验带隙（$\text{In}_2\text{O}_3$ 约 3.75、$\text{SnO}_2$ 约 3.6、ZnO 约 3.37、$\text{Ga}_2\text{O}_3$ 约 4.8 eV）对比，估算 PBE/实验的比值；
4. 按这个比值把阈值 3.0 eV 换算到 PBE 尺度，重跑漏斗，画出漏斗图并在“带隙 vs 形成能”平面上标出入围者；
5. **讨论**：修正后有几个已知 TCO 通过了？剩下没通过的是什么原因？与本节光伏的案例相比，TCO 这边的 PBE 偏差更大还是更小？

> **提示**：用 `Composition(f).reduced_composition` 做材料匹配，不要直接比化学式字符串——`Composition("CuInSe2").reduced_formula` 得到的是 `"InCuSe2"`，元素顺序会按电负性重排。

In [ ]:
# 在下方填写你的代码：


---

## 小结

| 核心概念 | 方法与实现 | 材料科学意义 |
|---|---|---|
| **布拉格定律与模拟 XRD** | `XRDCalculator(wavelength="CuKa").get_pattern(s)` | 峰位由晶胞参数定，峰强由结构因子定；连接计算结构与实验粉末图谱 |
| **系统消光** | 对比 NaCl / KCl / Si 的 $(111)$、$(200)$ | KCl 的 $(111)$ 因 $\text{K}^+$ 与 $\text{Cl}^-$ 等电子而熄灭；金刚石 Si 的 $(200)$ 被附加消光条件抹掉 |
| **物相鉴别** | 锐钛矿 $(101)$ 25.3° vs 金红石 $(110)$ 27.4° | 实验室每天都在做的事；Spurr–Myers 经验公式与 Rietveld 全谱拟合的分界 |
| **形成能与元素参考态** | 对每个元素取同素异形体的稳健最小值 | 一条脏的 `S8 = -23.6 eV/atom` 会污染全部含硫化合物；中位数 + 2 eV 容差可识别 |
| **热力学凸包** | `scipy.spatial.ConvexHull` 与 `pymatgen.PhaseDiagram` | **$E_f < 0$ 不等于稳定**；$E_{\text{hull}} < 25$ meV/atom 才算可合成；Magnéli 相 $\text{Ti}_5\text{O}_9$ 只高出约 7 meV |
| **Goldschmidt 容差因子** | Shannon 半径 + $t = (r_A + r_X)/\sqrt{2}(r_B + r_X)$ | 1926 年的几何判据在 1,929 条 DFT 数据上依然成立——但它是**窗口型**判据（窗口内外中位数差 3 倍），不是排序型预测器（$r = 0.07$） |
| **CBFV 组成特征** | Magpie 22 属性 $\times$ 5 统计量 = 110 维 | 只用化学式就能建模；模型自己学到“元素带隙与电负性最重要” |
| **带隙预测** | 随机森林，$R^2 \approx 0.83$、金属分类 AUC $\approx 0.97$ | PBE 带隙系统性低估实验值，模型学的是 PBE 不是实验 |
| **外推悬崖** | 留出整类化学 vs 随机划分 | $R^2$ 从 0.83 崩到 0.19（氟化物）。这是 `02b` Scaffold Split 教训在材料领域的翻版，而且更严重 |
| **高通量筛选漏斗** | 五道关卡，27,767 $\to$ 几百 | 判据必须有物理依据，且必须和数据的**尺度**对齐：直接套用实验带隙窗口会筛出一堆氟化物 |
| **健全性检验** | 用已知光伏材料回测漏斗 | 第一版漏斗七个已知材料**全军覆没**；诊断出 PBE 低估带隙后修正窗口才救回来。**筛不出已知答案的流程不可信** |

### 第六讲全景回顾

| 问题 | `06a` 的回答 | `06b` 的回答 |
|---|---|---|
| 晶体怎么表示？ | 晶格 + 基元 + 空间群，pymatgen `Structure` | — |
| 结构从哪来？ | CIF、COD（实验）、Materials Project（DFT） | — |
| 算出来的结构怎么和实验对上？ | — | 模拟 XRD 谱图与物相鉴别 |
| 这个材料能存在吗？ | — | 热力学凸包与 $E_{\text{hull}}$；Goldschmidt 容差因子 |
| 只知道化学式怎么建模？ | — | CBFV 组成特征向量 |
| 模型的预测能信吗？ | — | 留出整类化学：$R^2$ 0.83 $\to$ 0.19 |

### 下一讲预告

到这里，上半学期（第 01–06 讲）的**经典机器学习**部分就全部结束了。我们处理过的特征——分子描述符、指纹、组成统计量——都是**人工设计**的：是人决定了“该看电负性的加权平均”还是“该数芳环个数”。

但如果特征本身就该由数据来学呢？

下一讲 **`07 - Neural Networks and CNNs`（神经网络与材料显微视觉）** 进入下半学期的深度学习部分：从人工神经元和反向传播讲起，用 PyTorch 手写梯度下降，再用卷积神经网络直接从**显微图像**里自动提取特征——不再需要任何人工描述符。